# What is Llama thinking? Clusters of its rationales

Every answer in a steering run comes with a rationale. This notebook embeds Llama's **unsteered** rationales (10 per
comment, averaged), lays the comments out in 2D (UMAP), groups them (KMeans) and labels each group with its most typical
rationale words. Colours show the human score, Llama's prior score (its TOXIC share with no steering) and each comment's
best steering strength; hover a point to read the comment and one of Llama's rationales.

The embeddings come from `scripts/experiments/embed_rationales.py` (`<RUN>/rationale_embeddings.parquet`).

In [12]:
RUN = "../data/processed/civil-comments/consensus_n2000_train/steer_toxicity_llama3_8b_l10-25_grid"
SAMPLE = "../data/processed/civil-comments/consensus_n2000_train/sample.csv"
K = 5               # clusters
SEED = 0

In [13]:
import textwrap

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

scores = pd.read_csv(f"{RUN}/scores.csv")
ratings = pd.read_csv(f"{RUN}/ratings.csv")
sample = pd.read_csv(SAMPLE).set_index("id")
R = pd.read_parquet(f"{RUN}/rationale_embeddings.parquet")
ids = R.index.intersection(scores.id.unique()).sort_values()
R = R.loc[ids]

# per comment: human score, prior score (alpha 0), and the best alpha (lowest |LLM - human|; ties -> weakest push)
llm = scores.pivot_table(index="id", columns="weighting", values="llm_toxicity").loc[ids]
human = scores.groupby("id").human_toxicity.first().loc[ids]
dist = llm.sub(human, axis=0).abs()
alphas = dist.columns.to_numpy(dtype=float)
tied = np.isclose(dist.to_numpy(), dist.min(axis=1).to_numpy()[:, None])
best = np.array([alphas[t][np.abs(alphas[t]).argmin()] for t in tied])
movable = ~np.isclose(dist.to_numpy(), dist.to_numpy()[:, [0]]).all(axis=1)   # False: every alpha gives the same answer
unsteered = ratings[ratings.condition == "none"]
print(f"{len(ids)} comments, {len(alphas)} strengths; steering changes nothing for {(~movable).sum()}")

2000 comments, 15 strengths; steering changes nothing for 420


## Layout and clusters

UMAP (cosine) places comments with similar rationales near each other. KMeans groups the embeddings; each cluster's
label is its 5 most typical words (TF-IDF over the cluster's rationales, compared with the rest).

In [14]:
xy = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine", random_state=SEED).fit_transform(R.to_numpy())
cluster = KMeans(K, n_init=10, random_state=SEED).fit_predict(R.to_numpy())

text = unsteered.groupby("id").rationale.apply(" ".join).loc[ids]
tfidf = TfidfVectorizer(stop_words="english", max_df=0.5, min_df=5, ngram_range=(1, 2))
T = tfidf.fit_transform(text.values)
vocab = np.array(tfidf.get_feature_names_out())
words = {c: ", ".join(vocab[np.asarray(T[cluster == c].mean(axis=0)).ravel().argsort()[::-1][:5]]) for c in range(K)}

df = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "cluster": [f"{c + 1}" for c in cluster],
                   "human": human.values, "prior": llm[0.0].values, "best_alpha": best, "movable": movable}, index=ids)
wrap = lambda s, n=90: "<br>".join(textwrap.wrap(str(s), n)[:6])
df["comment"] = sample.comment_text.reindex(ids).map(wrap)
df["rationale"] = unsteered.groupby("id").rationale.first().loc[ids].map(wrap)

summary = df.groupby("cluster").agg(comments=("x", "size"), human=("human", "mean"), llama=("prior", "mean"))
summary["llama - human"] = summary.llama - summary.human
summary["words"] = [words[int(c) - 1] for c in summary.index]
summary.round(2)

/n/home12/jfontes/projects/ethos/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,comments,human,llama,llama - human,words
cluster,,,,,
1,223,0.28,0.14,-0.14,"playful, lighthearted, humorous, contain, does..."
2,478,0.53,0.79,0.26,"political, trump, divisive, strong, calling"
3,620,0.60,0.93,0.33,"person, strong, contains personal, profanity, ..."
4,353,0.35,0.10,-0.25,"comment expresses, contain, expresses, does co..."
5,326,0.56,0.95,0.39,"people, harmful, group, group people, stereotypes"


In [15]:
hover = {"x": False, "y": False, "comment": True, "rationale": True, "human": ":.2f", "prior": ":.2f",
         "best_alpha": ":+.2f", "cluster": True}                     # for the single-panel plots below
from plotly.subplots import make_subplots

custom = np.stack([df.comment, df.rationale, df.human, df.prior, df.best_alpha, df.cluster], axis=1)
hovertemplate = ("cluster %{customdata[5]}<br>%{customdata[0]}<br><br><i>%{customdata[1]}</i><br>"
                 "human %{customdata[2]:.2f}, Llama %{customdata[3]:.2f}, best α %{customdata[4]:+.2f}<extra></extra>")
fig = make_subplots(1, 2, horizontal_spacing=0.04,
                    subplot_titles=("Clusters (KMeans) and their typical words", "Llama's toxicity score (no steering)"))
colors = px.colors.qualitative.T10
for c in sorted(df.cluster.unique(), key=int):
    m = (df.cluster == c).to_numpy()
    fig.add_trace(go.Scatter(x=df.x[m], y=df.y[m], mode="markers", name=f"{c}: {words[int(c) - 1]}",
                             marker=dict(size=5, opacity=0.75, color=colors[(int(c) - 1) % len(colors)]),
                             customdata=custom[m], hovertemplate=hovertemplate, legendgroup="clusters"), 1, 1)
fig.add_trace(go.Scatter(x=df.x, y=df.y, mode="markers", showlegend=False, customdata=custom,
                         hovertemplate=hovertemplate,
                         marker=dict(size=5, opacity=0.8, color=df.prior, colorscale="RdYlBu_r", cmin=0, cmax=1,
                                     colorbar=dict(title="Llama<br>TOXIC share", x=1.0))), 1, 2)
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.update_layout(title="Llama's rationales: clusters next to Llama's toxicity score", width=1500, height=650,
                  legend=dict(title="cluster: typical words", orientation="h", y=-0.05, x=0))
fig.show()